# MIR systematic review — validated Gemini preliminary screening

**Scope:** 2020–2026 audio-based MIR studies relevant to vocal, ensemble/choral, instrumental, and general MIR tasks. This notebook screens **titles/abstracts/metadata only**. It does **not** perform full-text methodological quality appraisal.

**Safeguards:** conservative exclusion; evidence quoted from supplied records; two-shot fixed examples (eligible vs excluded); held-out human validation; resumable API calls; deterministic decision handling; separate human adjudication; PRISMA audit outputs. **Do not deploy automatic exclusions until validation is satisfactory.**

This is a proposed implementation, not a validated classifier out of the box. Your own human labels and validation results determine whether it is fit for use.

## 1. Install and configure

Run the next cell once, then restart the kernel if necessary. Set `GEMINI_API_KEY` in your environment or enter it privately when prompted. Avoid committing API keys to notebooks or repositories. Verify the selected model is available to your account.

In [7]:
%pip install --user -q -U google-genai pandas numpy scikit-learn tqdm openpyxl

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\python312\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import os, json, time, random, re, hashlib, getpass
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
from google import genai
from google.genai import types

INPUT_CSV = Path('mir_papers.csv')
OUT = Path('mir_screening_outputs'); OUT.mkdir(exist_ok=True)
MODEL = 'gemini-3.5-flash-lite'  # Recommended active model; alternatives: 'gemini-3.6-flash'
SEED = 2026
PILOT_N = 30
VALIDATION_N = 50
DELAY_SECONDS = 4.2  # 15 RPM free tier ceiling = 4.0s minimum delay to avoid 429 exhaustion
MAX_RETRIES = 5

api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            api_key = getpass.getpass('Gemini API key (hidden, press Enter to skip if testing): ').strip() or None
    except Exception:
        api_key = None

if api_key:
    client = genai.Client(api_key=api_key)
    print('GenAI client successfully initialized.')
else:
    client = None
    print('Notice: GEMINI_API_KEY is not set. API calls will be skipped until an API key is provided.')
print('Configured; input:', INPUT_CSV)


GenAI client successfully initialized.
Configured; input: mir_papers.csv


## 2. Load records and create stable IDs

Expected CSV columns: `title`, `abstract`, optionally `year`, `authors`, `venue`, `doi`, `record_id`. Case-insensitive alternatives are handled below. Empty abstracts are **human review**, not automatic exclusion. Export the sample template if you do not have a CSV yet.

In [9]:
if not INPUT_CSV.exists():
    pd.DataFrame([{'record_id':'R001','title':'Example: Audio piano transcription with neural networks', 'abstract':'We evaluate piano transcription on acoustic recordings using standard metrics.', 'year':2024, 'venue':'Example Venue', 'doi':''}]).to_csv(OUT/'input_template.csv',index=False)
    raise FileNotFoundError(f'Place your paper CSV at {INPUT_CSV}. Template: {OUT / "input_template.csv"}')

raw = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False).fillna('')
# Normalize column names (lowercase, strip whitespace, replace spaces with underscores)
raw.columns = [str(c).strip().lower().replace(' ','_') for c in raw.columns]
# Deduplicate columns: keep first occurrence (handles CSVs that have both 'Title' and 'title')
raw = raw.loc[:, ~raw.columns.duplicated(keep='first')]

# Alias mapping for common column variants across Scopus / WoS / Zotero exports
aliases = {
    'paper_title':'title', 'article_title':'title', 'summary':'abstract',
    'publication_year':'year', 'publication_title':'venue',
    'publication_venue':'venue', 'journal':'venue', 'conference':'venue',
    'paper_id':'record_id', 'id':'record_id'
}
for old, new in aliases.items():
    if old in raw.columns and new not in raw.columns:
        raw = raw.rename(columns={old: new})

assert 'title' in raw.columns and 'abstract' in raw.columns, 'CSV requires title and abstract columns.'

for c in ['year', 'authors', 'venue', 'doi']:
    if c not in raw.columns:
        raw[c] = ''

# PRISMA deduplication: deduplicate identical records by title and abstract
init_count = len(raw)
raw = raw.drop_duplicates(subset=['title', 'abstract'], keep='first').copy().reset_index(drop=True)

# Ensure record_id is clean and strictly unique
if 'record_id' not in raw.columns or raw['record_id'].nunique() != len(raw):
    raw['record_id'] = ['R' + str(i+1).zfill(5) for i in range(len(raw))]

raw['record_id'] = raw.record_id.astype(str)
assert raw.record_id.is_unique, 'record_id must be unique.'

raw['fingerprint'] = raw.apply(
    lambda r: hashlib.sha256((str(r['title']).strip().lower() + '|' + str(r['abstract']).strip().lower()).encode()).hexdigest()[:16],
    axis=1
)
print(f'Records loaded: {init_count} | Deduplicated records: {len(raw)}')
print(f'Missing abstracts: {sum(raw.abstract.str.strip() == "")} | Unique fingerprints: {raw.fingerprint.nunique()}')
raw.to_csv(OUT / 'records_normalized.csv', index=False)

Records loaded: 553 | Deduplicated records: 547
Missing abstracts: 0 | Unique fingerprints: 547


## 3. Screening criteria and fixed two-shot prompt

**Eligible to retrieve:** empirical or technical studies centrally addressing audio-based MIR, including audio-plus-score/text/multimodal approaches where music audio is substantively analyzed; relevant domains include singing, choir/ensemble, instruments, and general MIR. Examples: transcription, separation, pitch/onset/beat/key recognition, retrieval, alignment, performance analysis, and audio music representation learning. The **exact** eligibility scope must be approved by your research team before running.

**Clearly exclude:** symbolic-only MIDI/score work without music-audio analysis; unrelated fields, non-research content; out-of-window records if year reliably verified. **Do not exclude** for missing methodological detail in the abstract, venue prestige, missing code, or lack of limitations.

**Two-shot prompting structure:**
- **Shot 1 (Eligible - Audio MIR):** Demonstrates `RETRIEVE_FULL_TEXT` with verbatim evidence of acoustic audio processing.
- **Shot 2 (Excluded - Symbolic Only):** Demonstrates `EXCLUDE` (`SYMBOLIC_ONLY`) with verbatim evidence that no audio was processed.
- Ambiguous, short, or missing abstracts conservatively fall back to `HUMAN_REVIEW`.

In [10]:
SYSTEM_PROMPT = """You are a cautious systematic-review TITLE/ABSTRACT eligibility assistant for audio-based Music Information Retrieval (MIR).
Assess only the supplied title, abstract, and bibliographic metadata. Never invent facts or search outside the record.
Goal: minimize false exclusions. Do not evaluate paper quality, venue prestige, dataset rigor, code availability, or reproducibility from abstract omissions.
Scope: 2020–2026, technical/empirical music-audio analysis, including audio-based and audio-grounded multimodal methods for singing, ensemble/choral, instrumental, or general MIR.
Clearly exclude symbolic-only music methods without audio analysis, unrelated work, or non-research content. Exclude out-of-window only when year is explicitly present and unambiguous.
For ambiguous, short, or missing abstracts, choose HUMAN_REVIEW; if apparently relevant, choose RETRIEVE_FULL_TEXT.
Decision meanings:
- RETRIEVE_FULL_TEXT: plausible match requiring full-text eligibility assessment.
- EXCLUDE: explicit, unambiguous violation of a screening criterion.
- HUMAN_REVIEW: insufficient or conflicting information to decide safely.
Use exclusion_code only for EXCLUDE: YEAR_OUT_OF_RANGE, SYMBOLIC_ONLY, NOT_MIR, NOT_RESEARCH. Otherwise null.
Evidence must be a short verbatim quote from the supplied title/abstract/metadata or empty if none; never fabricate quotes.
Return exactly JSON with keys decision, exclusion_code, evidence, rationale, uncertainty, audio_relevance, domain.
uncertainty: LOW, MEDIUM, HIGH. audio_relevance: YES, NO, UNCLEAR. domain: VOCAL, ENSEMBLE, INSTRUMENTAL, GENERAL, UNCLEAR.

EXAMPLE 1 (Eligible - Audio MIR)
INPUT: {"title":"Neural Piano Transcription from Recordings","abstract":"We transcribe piano audio and evaluate note-level precision and recall on the MAESTRO dataset.","year":"2023","venue":"Conference"}
OUTPUT: {"decision":"RETRIEVE_FULL_TEXT","exclusion_code":null,"evidence":"transcribe piano audio","rationale":"Music-audio transcription is central.","uncertainty":"LOW","audio_relevance":"YES","domain":"INSTRUMENTAL"}

EXAMPLE 2 (Excluded - Symbolic Only)
INPUT: {"title":"Transformer-Based Symbolic Music Generation","abstract":"We generate MIDI note sequences from symbolic scores without processing audio.","year":"2024","venue":"Journal"}
OUTPUT: {"decision":"EXCLUDE","exclusion_code":"SYMBOLIC_ONLY","evidence":"without processing audio","rationale":"Explicitly symbolic-only with no music-audio analysis.","uncertainty":"LOW","audio_relevance":"NO","domain":"GENERAL"}
"""

FIELDS = ['record_id', 'title', 'abstract', 'year', 'authors', 'venue', 'doi']
ALLOWED = {'RETRIEVE_FULL_TEXT', 'EXCLUDE', 'HUMAN_REVIEW'}
CODES = {'YEAR_OUT_OF_RANGE', 'SYMBOLIC_ONLY', 'NOT_MIR', 'NOT_RESEARCH'}

def validate_response(result, record):
    if not isinstance(result, dict):
        raise ValueError('Response must be a JSON object')
    decision = str(result.get('decision', '')).strip().upper()
    result['decision'] = decision
    if decision not in ALLOWED:
        raise ValueError(f'Invalid decision: {decision}')
    code = result.get('exclusion_code')
    if code in ('', 'null', 'None', 'NONE', 'N/A', 'n/a', False):
        code = None
        result['exclusion_code'] = None
    elif code is not None:
        code = str(code).strip().upper()
        result['exclusion_code'] = code
    if decision == 'EXCLUDE' and code not in CODES:
        raise ValueError(f'Missing or invalid exclusion code for EXCLUDE: {code}')
    if decision != 'EXCLUDE' and code is not None:
        result['exclusion_code'] = None
    unc = str(result.get('uncertainty', '')).strip().upper()
    result['uncertainty'] = unc if unc in {'LOW', 'MEDIUM', 'HIGH'} else 'MEDIUM'
    ar = str(result.get('audio_relevance', '')).strip().upper()
    result['audio_relevance'] = ar if ar in {'YES', 'NO', 'UNCLEAR'} else 'UNCLEAR'
    dom = str(result.get('domain', '')).strip().upper()
    result['domain'] = dom if dom in {'VOCAL', 'ENSEMBLE', 'INSTRUMENTAL', 'GENERAL', 'UNCLEAR'} else 'UNCLEAR'
    evidence = str(result.get('evidence') or '').strip()
    source = ' '.join(str(record.get(k, '')) for k in ['title', 'abstract', 'year', 'authors', 'venue', 'doi'])
    if evidence and re.sub(r'\s+', ' ', evidence).casefold() not in re.sub(r'\s+', ' ', source).casefold():
        result['evidence_verbatim_match'] = False
    else:
        result['evidence_verbatim_match'] = True
    return result

def screen_record(record):
    payload = {k: str(record.get(k, '')) for k in FIELDS if k != 'record_id'}
    if not payload.get('abstract', '').strip():
        return {
            'decision': 'HUMAN_REVIEW',
            'exclusion_code': None,
            'evidence': '',
            'rationale': 'Missing abstract; human verification required.',
            'uncertainty': 'HIGH',
            'audio_relevance': 'UNCLEAR',
            'domain': 'UNCLEAR'
        }
    if client is None:
        return {
            'decision': 'HUMAN_REVIEW',
            'exclusion_code': None,
            'evidence': '',
            'rationale': 'GenAI client not initialized (GEMINI_API_KEY missing).',
            'uncertainty': 'HIGH',
            'audio_relevance': 'UNCLEAR',
            'domain': 'UNCLEAR',
            'api_error': True
        }
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents='INPUT: ' + json.dumps(payload, ensure_ascii=False) + '\nOUTPUT:',
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,
                    response_mime_type='application/json'
                )
            )
            raw_text = resp.text.strip() if resp.text else ''
            if raw_text.startswith('```json'):
                raw_text = raw_text[7:]
            elif raw_text.startswith('```'):
                raw_text = raw_text[3:]
            if raw_text.endswith('```'):
                raw_text = raw_text[:-3]
            return validate_response(json.loads(raw_text.strip()), record)
        except Exception as exc:
            if attempt == MAX_RETRIES - 1:
                return {
                    'decision': 'HUMAN_REVIEW',
                    'exclusion_code': None,
                    'evidence': '',
                    'rationale': f'API/validation error: {str(exc)[:240]}',
                    'api_error_detail': str(exc),
                    'uncertainty': 'HIGH',
                    'audio_relevance': 'UNCLEAR',
                    'domain': 'UNCLEAR',
                    'api_error': True
                }
            time.sleep(min(30, 2**attempt + random.random()))
        finally:
            time.sleep(DELAY_SECONDS)


## 4. Prepare human-labeled pilot and held-out validation sets

This notebook samples **30 pilot** and **50 held-out** records (adjust if needed). A human reviewer must label both sets independently of Gemini; preferably have two reviewers resolve disagreements. **Pilot labels are for revising the prompt. Held-out labels are for final validation only.** Do not change the prompt based on held-out results and then report that same test as independent validation.

In [11]:
rng = np.random.default_rng(SEED)
ids = raw.record_id.to_numpy()
order = rng.permutation(len(ids))
pilot_ids = set(ids[order[:min(PILOT_N, len(ids))]])
val_ids = set(ids[order[min(PILOT_N, len(ids)):min(PILOT_N + VALIDATION_N, len(ids))]])

for name, subset in [('pilot', pilot_ids), ('validation', val_ids)]:
    f = OUT / f'{name}_human_labels.csv'
    if not f.exists():
        d = raw[raw.record_id.isin(subset)][['record_id', 'title', 'abstract', 'year', 'venue']].copy()
        d['human_decision'] = ''
        d['human_exclusion_code'] = ''
        d['human_notes'] = ''
        d['reviewer_id'] = ''
        d.to_csv(f, index=False)
        print('Created', f, 'records:', len(d))
    else:
        print('Found existing', f)
print('Fill human_decision with RETRIEVE_FULL_TEXT, EXCLUDE, or HUMAN_REVIEW before validation.')

Found existing mir_screening_outputs\pilot_human_labels.csv
Found existing mir_screening_outputs\validation_human_labels.csv
Fill human_decision with RETRIEVE_FULL_TEXT, EXCLUDE, or HUMAN_REVIEW before validation.


## 5. Run Gemini with checkpointing

Start with the pilot (`RUN_MODE='pilot'`). After prompt calibration, run `validation`. Finally run `all` to screen the entire corpus. The notebook uses a JSONL cache keyed by record fingerprint **and prompt/model version**, preventing stale results from being silently reused after prompt changes. Re-running a mode resumes completed records.

In [12]:
RUN_MODE = 'pilot'  # Options: 'pilot', 'validation', 'all'
PROMPT_HASH = hashlib.sha256((SYSTEM_PROMPT + '|' + MODEL).encode()).hexdigest()[:16]
CACHE = OUT / 'gemini_screening_cache.jsonl'

def load_cache():
    data = {}
    if CACHE.exists():
        for line in CACHE.read_text(encoding='utf-8').splitlines():
            try:
                item = json.loads(line)
                # Ignore prior corrupted error entries from cache
                if item.get('api_error') or 'API/validation error' in item.get('rationale', ''):
                    continue
                data[(item['record_id'], item['fingerprint'], item['prompt_hash'])] = item
            except Exception:
                pass
    return data

def run_mode(mode):
    if mode not in {'pilot', 'validation', 'all'}:
        raise ValueError(f'Invalid mode: {mode}')
    if client is None:
        print('Error: Gemini API client is not configured. Set GEMINI_API_KEY in Section 1.')
        return None
    subset = raw if mode == 'all' else raw[raw.record_id.isin(pilot_ids if mode == 'pilot' else val_ids)]
    cache = load_cache()
    
    first_record_tested = False
    
    for _, r in tqdm(subset.iterrows(), total=len(subset), desc=mode):
        k = (r.record_id, r.fingerprint, PROMPT_HASH)
        if k in cache:
            continue
            
        result = screen_record(r.to_dict())
        
        # PRECAUTION: Check if the first row in this run already encounters an API error
        if not first_record_tested:
            first_record_tested = True
            if result.get('api_error'):
                err_detail = result.get('api_error_detail') or result.get('rationale', 'Unknown API error')
                print('\n' + '=' * 75)
                print(f'[PRECAUTION TRIGGERED: CANCELING RUN ON FIRST RECORD]')
                print(f'Record ID: {r.record_id}')
                print(f'Error message: {err_detail}')
                print('=' * 75)
                print('Execution cancelled immediately to prevent burning API quota or polluting the cache.')
                print('Recommended actions:')
                print(' 1. Check if GEMINI_API_KEY is active and valid.')
                print(' 2. Use a high-availability model like MODEL = \'gemini-3.5-flash-lite\'.')
                print(' 3. Ensure DELAY_SECONDS >= 4.2 to comply with 15 RPM free tier limits.')
                print('=' * 75 + '\n')
                raise RuntimeError(f'API error on first record ({r.record_id}): {err_detail}')

        # Only cache valid screening results (never cache API failures)
        if not result.get('api_error'):
            item = {'record_id': r.record_id, 'fingerprint': r.fingerprint, 'prompt_hash': PROMPT_HASH, 'model': MODEL, **result}
            with CACHE.open('a', encoding='utf-8') as f:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
            cache[k] = item
        else:
            print(f'Warning: Record {r.record_id} encountered API error: {result.get("rationale")}')

    valid_items = [cache[(r.record_id, r.fingerprint, PROMPT_HASH)] for _, r in subset.iterrows() if (r.record_id, r.fingerprint, PROMPT_HASH) in cache]
    result_df = pd.DataFrame(valid_items)
    out = OUT / f'gemini_{mode}_results.csv'
    result_df.to_csv(out, index=False)
    print('Saved:', out, '| Decisions:', result_df.decision.value_counts().to_dict() if not result_df.empty else 'No valid decisions saved')
    return result_df

if client is not None:
    results = run_mode(RUN_MODE)
else:
    print('Notice: Set your GEMINI_API_KEY in Section 1 to run Gemini screening.')
    results = None


pilot: 100%|██████████| 30/30 [12:11<00:00, 24.37s/it]

Saved: mir_screening_outputs\gemini_pilot_results.csv | Decisions: {'HUMAN_REVIEW': 30}


## 6. Held-out validation — safety first

Human decisions are the reference standard. Treat `RETRIEVE_FULL_TEXT` **and** `HUMAN_REVIEW` as potentially eligible. The most consequential error is **Gemini EXCLUDE when the human reviewer would retrieve or review**. We report that unsafe-exclusion count, its rate among human-retained records, and recall for retaining potentially eligible studies. A sample of 50 provides only a preliminary estimate; uncertainty can remain substantial even with zero observed misses.

In [13]:
def evaluate(mode='validation'):
    human_path = OUT / f'{mode}_human_labels.csv'
    pred_path = OUT / f'gemini_{mode}_results.csv'
    if not human_path.exists():
        print(f'Human labels file not found: {human_path}')
        return None
    if not pred_path.exists():
        print(f"No model predictions found at {pred_path}. Run RUN_MODE='{mode}' first.")
        return None
        
    human = pd.read_csv(human_path, dtype=str, keep_default_na=False)
    pred = pd.read_csv(pred_path, dtype=str, keep_default_na=False)
    d = human.merge(pred[['record_id', 'decision', 'exclusion_code', 'rationale', 'evidence']], on='record_id', validate='one_to_one')
    d = d[d.human_decision.isin(ALLOWED)].copy()
    if not len(d):
        print('No labeled records yet. Fill in human decisions in', human_path)
        return None
    d['human_retain'] = d.human_decision != 'EXCLUDE'
    d['model_retain'] = d.decision != 'EXCLUDE'
    d['unsafe_exclusion'] = d.human_retain & ~d.model_retain
    positives = int(d.human_retain.sum())
    unsafe = int(d.unsafe_exclusion.sum())
    print('Human-labeled records:', len(d), '| Human retain/review:', positives)
    print('Unsafe exclusions:', unsafe, '| Eligible-retention recall:', (positives - unsafe) / positives if positives else 'undefined')
    print('Unsafe exclusion rate among human-retained:', unsafe / positives if positives else 'undefined')
    print('Human vs model decision table:\n', pd.crosstab(d.human_decision, d.decision, dropna=False))
    print('3-class Cohen kappa:', cohen_kappa_score(d.human_decision, d.decision) if len(d) > 1 else 'undefined')
    d.to_csv(OUT / f'{mode}_validation_audit.csv', index=False)
    d[d.unsafe_exclusion].to_csv(OUT / f'{mode}_unsafe_exclusions.csv', index=False)
    return d

# After filling validation_human_labels.csv and running RUN_MODE='validation':
# audit = evaluate('validation')

## 7. Human adjudication and final screening export

Before claiming that exclusions are safe, manually inspect **every proposed EXCLUDE**, all `HUMAN_REVIEW`, all API errors, and a random sample of proposed retrieves. This conservative workflow avoids treating the LLM as an autonomous eligibility reviewer. Fill `final_human_decision` after review. Keep original AI decisions unchanged for audit. For PRISMA, count exclusions only after this final decision.

In [14]:
def make_adjudication_sheet():
    allpred = run_mode('all')
    if allpred is None or len(allpred) == 0:
        print('Run mode all returned no results. Check API key and configuration.')
        return None
    merged = raw.merge(allpred.drop(columns=['fingerprint'], errors='ignore'), on='record_id', validate='one_to_one')
    review = merged.copy()
    review['requires_human_review'] = review.decision.isin(['EXCLUDE', 'HUMAN_REVIEW']) | review.get('api_error', pd.Series(False, index=review.index)).fillna(False).astype(bool)
    review['final_human_decision'] = ''
    review['final_exclusion_reason'] = ''
    review['reviewer_id'] = ''
    review['review_notes'] = ''
    path = OUT / 'human_adjudication_queue.csv'
    review.to_csv(path, index=False)
    print('Saved', path, '| Required human reviews:', int(review.requires_human_review.sum()))
    return review

# After final validation and approval: adjudication = make_adjudication_sheet()

## 8. Full-text appraisal plan (do NOT run on abstracts)

Once PDFs are obtained, first apply full-text eligibility. Then use a separate, evidence-based appraisal sheet. Below is a **provisional** five-item rubric, not a validated instrument. Confirm the rubric and thresholds with your supervisor and document deviations from any cited source.

| Item | Requires | Score guidance |
|---|---|---|
| C1 Publication verification | Publisher/index/venue metadata | 0–2; verify peer-review status; avoid assuming rank equals methodological quality |
| C2 Methodological rigor | Methods, experimental setup, evaluation | 0–2; metrics, baselines, documented design |
| C3 Dataset appropriateness | Dataset/annotation and splits sections | 0–2; relevant and adequately described data |
| C4 Topical centrality | Full text + RQ mapping | 0–2; centrality of audio MIR to RQs |
| C5 Contribution and reproducibility | Results, limitations, artifacts | 0–2; contribution and transparency; absence of public code is not automatically fatal |

Score each item with a **quoted supporting passage and PDF page**; use `INSUFFICIENT_EVIDENCE` when uncertain rather than inventing a score. Two human reviewers should independently appraise a sample and resolve disagreements. Calculate 0–10 totals only when all five scores are supported. Keep appraisal separate from study eligibility; do not exclude an otherwise relevant study solely because it is Q3/Q4, lacks code, or replicates prior work.

The uploaded reference paper uses a **different two-domain 45-point rubric** (Methodological Quality 20; Relevance Quality 25), adapted there from **Borbon and Ebardo (2025)**. The C1–C5 / 10-point version is a proposed adaptation and should not be attributed to that source as though it were identical.

PRISMA 2020: record counts at identification, screening, full-text retrieval, eligibility, and inclusion; report AI model/version, prompt, examples, selection rules, human oversight, and validation results. An AI `EXCLUDE` label is not itself a completed PRISMA exclusion unless the workflow explicitly uses it as such.

In [15]:
appraisal_cols = ['record_id', 'pdf_path', 'full_text_eligible', 'full_text_exclusion_reason', 'reviewer_id']
for i in range(1, 6):
    appraisal_cols += [f'C{i}_score', f'C{i}_evidence_quote', f'C{i}_pdf_page', f'C{i}_notes']
appraisal_cols += ['total_score', 'appraisal_decision', 'adjudication_notes']
pd.DataFrame(columns=appraisal_cols).to_csv(OUT / 'full_text_quality_appraisal_template.csv', index=False)
print('Created full-text appraisal template. All scoring requires full-text evidence.')

Created full-text appraisal template. All scoring requires full-text evidence.


## 7. Finish remaining papers in safe batches & run SLR Phases 3 to 5

This cell re-screens the 108 records that previously hit API limits in 6 rate-limited batches (14 RPM, `gemini-3.5-flash-lite`). Once complete, it automatically executes SLR Phases 3 (NLP/LSA), 4 (Topic Benchmarking), and 5 (Network Synthesis).

In [ ]:
import os, getpass
from screen_and_run_pipeline import run_batch_screening

# Check local variable, environment variable, or prompt interactively
key = globals().get('api_key') or os.getenv('GEMINI_API_KEY')
if not key:
    key = getpass.getpass('Gemini API key: ').strip() or None

if key:
    api_key = key
    run_batch_screening(key)
else:
    print('Error: GEMINI_API_KEY is required to run screening.')
